# GCI In-Class Competition: Home Credit Default Risk
## Notebook 3: Ensemble Training & Stacking

**Objective**: Train 3 models with best params, create OOF predictions, stack with Ridge meta-learner
**Method**: 5-fold Stratified CV for each model, Ridge regression on OOF
**Hardware**: Google Colab T4 GPU
**Prerequisite**: Run Notebook 1 & 2 first

---

### Ensemble Strategy
1. **Train 3 models** with best Optuna params using 5-fold CV
2. **Collect OOF predictions** from each model
3. **Apply target encoding** WITHIN each fold (no leakage!)
4. **Stack with Ridge meta-learner** (5-fold CV on OOF)
5. **Optimize blend weights** on hold-out validation set
6. **Feature selection** via SHAP on best single model
7. **Retrain final ensemble** on full data for submission

In [ ]:
# 1. Setup and Library Imports
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import RidgeCV, LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
import shap
import time
import os
import gc
import json
import joblib
import warnings
warnings.filterwarnings('ignore')

print(f"LightGBM: {lgb.__version__}")
print(f"XGBoost: {xgb.__version__}")
print(f"CatBoost: {cb.__version__}")
print(f"SHAP: {shap.__version__}")

In [ ]:
# 2. Mount Drive & Load Data
from google.colab import drive
drive.mount('/content/drive')

WORK_DIR = '/content/drive/MyDrive/gci_competition'

# Load engineered datasets
train = pd.read_parquet(f'{WORK_DIR}/features/train_engineered.parquet')
test = pd.read_parquet(f'{WORK_DIR}/features/test_engineered.parquet')

# Load metadata
with open(f'{WORK_DIR}/features/feature_cols.json', 'r') as f:
    feature_cols = json.load(f)

y = np.load(f'{WORK_DIR}/features/y_train.npy')

with open(f'{WORK_DIR}/features/high_card_cols.json', 'r') as f:
    high_card_cols = json.load(f)

with open(f'{WORK_DIR}/features/high_corr_features.json', 'r') as f:
    high_corr_features = json.load(f)

with open(f'{WORK_DIR}/optuna_studies/all_best_params.json', 'r') as f:
    all_best_params = json.load(f)

# Prepare X
X = train[feature_cols].copy()
X_test = test[feature_cols].copy()
X = X.fillna(-999)
X_test = X_test.fillna(-999)

print(f"Train shape: {X.shape}")
print(f"Test shape: {X_test.shape}")
print(f"Features: {len(feature_cols)}")
print(f"Target shape: {y.shape}")
print(f"Positive rate: {y.mean():.4f}")

In [ ]:
# 3. Categorical Feature Indices
original_cats = [
    'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'NAME_INCOME_TYPE',
    'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE',
    'OCCUPATION_TYPE', 'ORGANIZATION_TYPE', 'FLAG_MOBIL',
    'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE'
]

cat_indices = [feature_cols.index(c) for c in original_cats if c in feature_cols]
print(f"Categorical feature indices: {len(cat_indices)}")
for idx in cat_indices:
    print(f"  [{idx}] {feature_cols[idx]}")

In [ ]:
# 4. Target Encoding Function (CV-Safe)

def apply_target_encoding_cv(X_train, X_val, X_test, y_train, cat_cols, smoothing=10):
    """
    Apply target encoding within CV fold to prevent leakage.
    Returns encoded X_train, X_val, X_test.
    """
    X_tr = X_train.copy()
    X_va = X_val.copy()
    X_te = X_test.copy()
    
    global_mean = y_train.mean()
    
    for col in cat_cols:
        if col not in X_tr.columns:
            continue
            
        # Compute smoothed means on training fold only
        cat_means = y_train.groupby(X_tr[col]).mean()
        cat_counts = X_tr[col].value_counts()
        smoothed = (cat_means * cat_counts + global_mean * smoothing) / (cat_counts + smoothing)
        
        # Apply to train, val, test
        X_tr[f'{col}_TE'] = X_tr[col].map(smoothed).fillna(global_mean)
        X_va[f'{col}_TE'] = X_va[col].map(smoothed).fillna(global_mean)
        X_te[f'{col}_TE'] = X_te[col].map(smoothed).fillna(global_mean)
    
    return X_tr, X_va, X_te

# Only apply to high-cardinality categoricals that exist in features
te_cols = [c for c in high_card_cols if c in feature_cols]
print(f"Target encoding columns: {te_cols}")

In [ ]:
# 5. Create Hold-Out Validation Set (10% stratified)
from sklearn.model_selection import train_test_split

HOLDOUT_SIZE = 0.1
RANDOM_STATE = 42

X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(
    X, y, test_size=HOLDOUT_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f"Full train: {X_train_full.shape}")
print(f"Holdout: {X_holdout.shape}")
print(f"Holdout positive rate: {y_holdout.mean():.4f}")


In [ ]:
# 6. Train LightGBM with 5-Fold CV

N_SPLITS = 5
EARLY_STOPPING_ROUNDS = 100
N_ESTIMATORS = 3000
SCALE_POS_WEIGHT = (1 - y.mean()) / y.mean()

lgb_params = all_best_params['lightgbm'].copy()
lgb_params.update({
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'device': 'gpu',
    'gpu_platform_id': 0,
    'gpu_device_id': 0,
    'scale_pos_weight': SCALE_POS_WEIGHT,
    'random_state': RANDOM_STATE,
    'n_jobs': 2,
    'verbose': -1
})

# OOF predictions
lgb_oof = np.zeros(len(X_train_full))
lgb_test_preds = np.zeros(len(X_test))
lgb_models = []
lgb_feature_importance = []

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

print("=" * 60)
print("Training LightGBM (5-fold CV)")
print("=" * 60)

t0 = time.time()
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_full, y_train_full)):
    fold_start = time.time()
    
    X_tr = X_train_full.iloc[train_idx].copy()
    y_tr = y_train_full[train_idx]
    X_va = X_train_full.iloc[val_idx].copy()
    y_va = y_train_full[val_idx]
    
    # Apply target encoding within fold
    X_tr, X_va, X_test_fold = apply_target_encoding_cv(
        X_tr, X_va, X_test, y_tr, te_cols
    )
    
    model = lgb.LGBMClassifier(**lgb_params, n_estimators=N_ESTIMATORS)
    
    callbacks = [
        lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False),
        lgb.log_evaluation(period=100)
    ]
    
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric='auc',
        categorical_feature=cat_indices,
        callbacks=callbacks
    )
    
    val_pred = model.predict_proba(X_va)[:, 1]
    lgb_oof[val_idx] = val_pred
    lgb_test_preds += model.predict_proba(X_test_fold)[:, 1] / N_SPLITS
    
    fold_auc = roc_auc_score(y_va, val_pred)
    lgb_models.append(model)
    lgb_feature_importance.append(model.feature_importances_)
    
    print(f"  Fold {fold+1}: AUC = {fold_auc:.5f}, Iter = {model.best_iteration_}, Time = {time.time()-fold_start:.1f}s")

lgb_cv_auc = roc_auc_score(y_train_full, lgb_oof)
print(f"\nLightGBM 5-Fold CV AUC: {lgb_cv_auc:.5f}")
print(f"Total time: {time.time()-t0:.1f}s")

# Save OOF and models
np.save(f'{WORK_DIR}/models/lgb_oof.npy', lgb_oof)
np.save(f'{WORK_DIR}/models/lgb_test_preds.npy', lgb_test_preds)
joblib.dump(lgb_models, f'{WORK_DIR}/models/lgb_models.pkl')

In [ ]:
# 7. Train XGBoost with 5-Fold CV

xgb_params = all_best_params['xgboost'].copy()
xgb_params.update({
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'tree_method': 'gpu_hist',
    'gpu_id': 0,
    'scale_pos_weight': SCALE_POS_WEIGHT,
    'random_state': RANDOM_STATE,
    'n_jobs': 2,
    'verbosity': 0,
    'enable_categorical': True
})

xgb_oof = np.zeros(len(X_train_full))
xgb_test_preds = np.zeros(len(X_test))
xgb_models = []

print("=" * 60)
print("Training XGBoost (5-fold CV)")
print("=" * 60)

t0 = time.time()
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_full, y_train_full)):
    fold_start = time.time()
    
    X_tr = X_train_full.iloc[train_idx].copy()
    y_tr = y_train_full[train_idx]
    X_va = X_train_full.iloc[val_idx].copy()
    y_va = y_train_full[val_idx]
    
    # Apply target encoding within fold
    X_tr, X_va, X_test_fold = apply_target_encoding_cv(
        X_tr, X_va, X_test, y_tr, te_cols
    )
    
    dtrain = xgb.DMatrix(X_tr, label=y_tr, enable_categorical=True)
    dval = xgb.DMatrix(X_va, label=y_va, enable_categorical=True)
    dtest = xgb.DMatrix(X_test_fold, enable_categorical=True)
    
    model = xgb.train(
        xgb_params,
        dtrain,
        num_boost_round=N_ESTIMATORS,
        evals=[(dval, 'val')],
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        verbose_eval=100
    )
    
    val_pred = model.predict(dval, iteration_range=(0, model.best_iteration + 1))
    xgb_oof[val_idx] = val_pred
    xgb_test_preds += model.predict(dtest, iteration_range=(0, model.best_iteration + 1)) / N_SPLITS
    
    fold_auc = roc_auc_score(y_va, val_pred)
    xgb_models.append(model)
    
    print(f"  Fold {fold+1}: AUC = {fold_auc:.5f}, Iter = {model.best_iteration}, Time = {time.time()-fold_start:.1f}s")

xgb_cv_auc = roc_auc_score(y_train_full, xgb_oof)
print(f"\nXGBoost 5-Fold CV AUC: {xgb_cv_auc:.5f}")
print(f"Total time: {time.time()-t0:.1f}s")

np.save(f'{WORK_DIR}/models/xgb_oof.npy', xgb_oof)
np.save(f'{WORK_DIR}/models/xgb_test_preds.npy', xgb_test_preds)
joblib.dump(xgb_models, f'{WORK_DIR}/models/xgb_models.pkl')

In [ ]:
# 8. Train CatBoost with 5-Fold CV

cat_params = all_best_params['catboost'].copy()
cat_params.update({
    'objective': 'Logloss',
    'eval_metric': 'AUC',
    'task_type': 'GPU',
    'devices': '0',
    'scale_pos_weight': SCALE_POS_WEIGHT,
    'random_seed': RANDOM_STATE,
    'verbose': False,
    'allow_writing_files': False
})

cat_oof = np.zeros(len(X_train_full))
cat_test_preds = np.zeros(len(X_test))
cat_models = []

print("=" * 60)
print("Training CatBoost (5-fold CV)")
print("=" * 60)

t0 = time.time()
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_full, y_train_full)):
    fold_start = time.time()
    
    X_tr = X_train_full.iloc[train_idx].copy()
    y_tr = y_train_full[train_idx]
    X_va = X_train_full.iloc[val_idx].copy()
    y_va = y_train_full[val_idx]
    
    # Apply target encoding within fold
    X_tr, X_va, X_test_fold = apply_target_encoding_cv(
        X_tr, X_va, X_test, y_tr, te_cols
    )
    
    model = cb.CatBoostClassifier(**cat_params, iterations=N_ESTIMATORS)
    
    model.fit(
        X_tr, y_tr,
        eval_set=(X_va, y_va),
        cat_features=cat_indices,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        verbose=100
    )
    
    val_pred = model.predict_proba(X_va)[:, 1]
    cat_oof[val_idx] = val_pred
    cat_test_preds += model.predict_proba(X_test_fold)[:, 1] / N_SPLITS
    
    fold_auc = roc_auc_score(y_va, val_pred)
    cat_models.append(model)
    
    print(f"  Fold {fold+1}: AUC = {fold_auc:.5f}, Iter = {model.best_iteration_}, Time = {time.time()-fold_start:.1f}s")

cat_cv_auc = roc_auc_score(y_train_full, cat_oof)
print(f"\nCatBoost 5-Fold CV AUC: {cat_cv_auc:.5f}")
print(f"Total time: {time.time()-t0:.1f}s")

np.save(f'{WORK_DIR}/models/cat_oof.npy', cat_oof)
np.save(f'{WORK_DIR}/models/cat_test_preds.npy', cat_test_preds)
joblib.dump(cat_models, f'{WORK_DIR}/models/cat_models.pkl')

In [ ]:
# 9. Stacking with Ridge Meta-Learner

# Combine OOF predictions
oof_stack = np.column_stack([lgb_oof, xgb_oof, cat_oof])
test_stack = np.column_stack([lgb_test_preds, xgb_test_preds, cat_test_preds])

print(f"OOF stack shape: {oof_stack.shape}")
print(f"Test stack shape: {test_stack.shape}")

# RidgeCV on OOF (inner CV for alpha selection)
ridge = RidgeCV(alphas=[0.001, 0.01, 0.1, 1.0, 10.0, 100.0], cv=5)
ridge.fit(oof_stack, y_train_full)

print(f"Best alpha: {ridge.alpha_}")
print(f"Ridge coefficients: {ridge.coef_}")
print(f"Ridge intercept: {ridge.intercept_}")

# OOF predictions from meta-learner
meta_oof = ridge.predict(oof_stack)
meta_auc = roc_auc_score(y_train_full, meta_oof)
print(f"\nMeta-learner (Ridge) OOF AUC: {meta_auc:.5f}")

# Hold-out evaluation
holdout_lgb = np.zeros(len(X_holdout))
holdout_xgb = np.zeros(len(X_holdout))
holdout_cat = np.zeros(len(X_holdout))

for model in lgb_models:
    holdout_lgb += model.predict_proba(X_holdout)[:, 1] / len(lgb_models)

for model in xgb_models:
    dhold = xgb.DMatrix(X_holdout, enable_categorical=True)
    holdout_xgb += model.predict(dhold, iteration_range=(0, model.best_iteration + 1)) / len(xgb_models)

for model in cat_models:
    holdout_cat += model.predict_proba(X_holdout)[:, 1] / len(cat_models)

holdout_stack = np.column_stack([holdout_lgb, holdout_xgb, holdout_cat])
holdout_meta = ridge.predict(holdout_stack)
holdout_auc = roc_auc_score(y_holdout, holdout_meta)
print(f"Hold-out AUC (Ridge): {holdout_auc:.5f}")

# Also try LogisticRegression meta-learner
logreg = LogisticRegressionCV(Cs=[0.001, 0.01, 0.1, 1.0, 10.0, 100.0], cv=5, random_state=RANDOM_STATE, max_iter=1000)
logreg.fit(oof_stack, y_train_full)

logreg_oof = logreg.predict_proba(oof_stack)[:, 1]
logreg_auc = roc_auc_score(y_train_full, logreg_oof)
print(f"Meta-learner (LogReg) OOF AUC: {logreg_auc:.5f}")

holdout_logreg = logreg.predict_proba(holdout_stack)[:, 1]
holdout_logreg_auc = roc_auc_score(y_holdout, holdout_logreg)
print(f"Hold-out AUC (LogReg): {holdout_logreg_auc:.5f}")

# Choose best meta-learner
if holdout_logreg_auc > holdout_auc:
    best_meta = logreg
    best_meta_name = 'LogisticRegression'
    meta_test_preds = logreg.predict_proba(test_stack)[:, 1]
else:
    best_meta = ridge
    best_meta_name = 'Ridge'
    meta_test_preds = ridge.predict(test_stack)

print(f"\nBest meta-learner: {best_meta_name}")

# Save meta-learner
joblib.dump(best_meta, f'{WORK_DIR}/models/meta_learner.pkl')
np.save(f'{WORK_DIR}/models/meta_test_preds.npy', meta_test_preds)

In [ ]:
# 10. Optimize Blend Weights on Hold-Out

# Simple average
avg_holdout = (holdout_lgb + holdout_xgb + holdout_cat) / 3
avg_auc = roc_auc_score(y_holdout, avg_holdout)
print(f"Simple average holdout AUC: {avg_auc:.5f}")

# Grid search for optimal weights
best_weight = None
best_auc = 0

for w1 in np.arange(0.2, 0.6, 0.05):
    for w2 in np.arange(0.2, 0.6, 0.05):
        w3 = 1 - w1 - w2
        if w3 < 0.1 or w3 > 0.6:
            continue
        blend = w1 * holdout_lgb + w2 * holdout_xgb + w3 * holdout_cat
        auc = roc_auc_score(y_holdout, blend)
        if auc > best_auc:
            best_auc = auc
            best_weight = (w1, w2, w3)

print(f"\nBest blend weights: LGB={best_weight[0]:.2f}, XGB={best_weight[1]:.2f}, CAT={best_weight[2]:.2f}")
print(f"Best blend holdout AUC: {best_auc:.5f}")

# Also evaluate meta-learner predictions
print(f"Meta-learner holdout AUC: {holdout_auc if best_meta_name == 'Ridge' else holdout_logreg_auc:.5f}")

# Choose final blend strategy
if best_auc > max(holdout_auc, holdout_logreg_auc):
    final_strategy = 'weighted_average'
    final_weights = best_weight
    final_test = best_weight[0] * lgb_test_preds + best_weight[1] * xgb_test_preds + best_weight[2] * cat_test_preds
else:
    final_strategy = 'meta_learner'
    final_weights = None
    final_test = meta_test_preds

print(f"\nFinal strategy: {final_strategy}")
if final_weights:
    print(f"Final weights: {final_weights}")

# Save final test predictions
np.save(f'{WORK_DIR}/models/final_test_preds.npy', final_test)
with open(f'{WORK_DIR}/models/final_strategy.json', 'w') as f:
    json.dump({'strategy': final_strategy, 'weights': final_weights}, f)

In [ ]:
# 11. Feature Selection via SHAP (on best single model)

# Determine best single model
single_aucs = {'LightGBM': lgb_cv_auc, 'XGBoost': xgb_cv_auc, 'CatBoost': cat_cv_auc}
best_model_name = max(single_aucs, key=single_aucs.get)
print(f"Best single model: {best_model_name} (AUC: {single_aucs[best_model_name]:.5f})")

# Use best model's first fold for SHAP (representative)
if best_model_name == 'LightGBM':
    best_model = lgb_models[0]
    X_sample = X_train_full.sample(n=min(5000, len(X_train_full)), random_state=42)
    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_sample)
elif best_model_name == 'XGBoost':
    best_model = xgb_models[0]
    X_sample = X_train_full.sample(n=min(5000, len(X_train_full)), random_state=42)
    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_sample)
else:
    best_model = cat_models[0]
    X_sample = X_train_full.sample(n=min(5000, len(X_train_full)), random_state=42)
    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_sample)

# Mean absolute SHAP values
if isinstance(shap_values, list):
    shap_values = shap_values[1]  # Positive class
mean_shap = np.abs(shap_values).mean(axis=0)

# Feature importance dataframe
shap_df = pd.DataFrame({
    'feature': feature_cols,
    'mean_shap': mean_shap
}).sort_values('mean_shap', ascending=False)

print("\nTop 30 features by SHAP importance:")
print(shap_df.head(30))

# Features to drop (near-zero SHAP)
shap_threshold = 0.001
low_shap_features = shap_df[shap_df['mean_shap'] < shap_threshold]['feature'].tolist()
print(f"\nFeatures with SHAP < {shap_threshold}: {len(low_shap_features)}")

# Save SHAP results
shap_df.to_csv(f'{WORK_DIR}/models/shap_importance.csv', index=False)
with open(f'{WORK_DIR}/models/low_shap_features.json', 'w') as f:
    json.dump(low_shap_features, f)

In [ ]:
# 12. Retrain Final Models on Full Training Data

# Prepare full training data with target encoding
X_full = X.copy()
X_test_full = X_test.copy()

# Apply target encoding on full data for final models
global_mean = y.mean()
for col in te_cols:
    if col in X_full.columns:
        cat_means = y.groupby(X_full[col]).mean()
        cat_counts = X_full[col].value_counts()
        smoothing = 10
        smoothed = (cat_means * cat_counts + global_mean * smoothing) / (cat_counts + smoothing)
        
        X_full[f'{col}_TE'] = X_full[col].map(smoothed).fillna(global_mean)
        X_test_full[f'{col}_TE'] = X_test_full[col].map(smoothed).fillna(global_mean)

print(f"Full train shape after TE: {X_full.shape}")
print(f"Full test shape after TE: {X_test_full.shape}")

# Retrain LightGBM on full data
print("\nRetraining LightGBM on full data...")
lgb_final = lgb.LGBMClassifier(**lgb_params, n_estimators=3000)
lgb_final.fit(X_full, y, categorical_feature=cat_indices)
lgb_final_preds = lgb_final.predict_proba(X_test_full)[:, 1]
joblib.dump(lgb_final, f'{WORK_DIR}/models/lgb_final.pkl')

# Retrain XGBoost on full data
print("Retraining XGBoost on full data...")
dtrain_full = xgb.DMatrix(X_full, label=y, enable_categorical=True)
dtest_full = xgb.DMatrix(X_test_full, enable_categorical=True)
xgb_final = xgb.train(xgb_params, dtrain_full, num_boost_round=lgb_final.best_iteration_ if hasattr(lgb_final, 'best_iteration_') else 1000)
xgb_final_preds = xgb_final.predict(dtest_full)
joblib.dump(xgb_final, f'{WORK_DIR}/models/xgb_final.pkl')

# Retrain CatBoost on full data
print("Retraining CatBoost on full data...")
cat_final = cb.CatBoostClassifier(**cat_params, iterations=1000)
cat_final.fit(X_full, y, cat_features=cat_indices, verbose=100)
cat_final_preds = cat_final.predict_proba(X_test_full)[:, 1]
joblib.dump(cat_final, f'{WORK_DIR}/models/cat_final.pkl')

# Final ensemble prediction
if final_strategy == 'weighted_average':
    final_preds = final_weights[0] * lgb_final_preds + final_weights[1] * xgb_final_preds + final_weights[2] * cat_final_preds
else:
    final_stack = np.column_stack([lgb_final_preds, xgb_final_preds, cat_final_preds])
    if best_meta_name == 'Ridge':
        final_preds = best_meta.predict(final_stack)
    else:
        final_preds = best_meta.predict_proba(final_stack)[:, 1]

np.save(f'{WORK_DIR}/models/final_full_preds.npy', final_preds)
print(f"\nFinal ensemble predictions saved!")
print(f"Shape: {final_preds.shape}")
print(f"Mean: {final_preds.mean():.4f}")
print(f"Std: {final_preds.std():.4f}")
print(f"Min: {final_preds.min():.4f}")
print(f"Max: {final_preds.max():.4f}")

In [ ]:
# 13. Summary

print("=" * 60)
print("ENSEMBLE TRAINING SUMMARY")
print("=" * 60)

print(f"\n5-Fold CV AUCs:")
print(f"  LightGBM:  {lgb_cv_auc:.5f}")
print(f"  XGBoost:   {xgb_cv_auc:.5f}")
print(f"  CatBoost:  {cat_cv_auc:.5f}")
print(f"  Meta-learner ({best_meta_name}): {max(holdout_auc, holdout_logreg_auc):.5f} (holdout)")
print(f"  Weighted avg: {best_auc:.5f} (holdout)")
print(f"  Simple avg: {avg_auc:.5f} (holdout)")

print(f"\nBest single model: {best_model_name}")
print(f"Final strategy: {final_strategy}")
if final_weights:
    print(f"Final weights: LGB={final_weights[0]:.2f}, XGB={final_weights[1]:.2f}, CAT={final_weights[2]:.2f}")

print(f"\nLow SHAP features to consider dropping: {len(low_shap_features)}")

print("\n✅ Notebook 3 Complete! Ready for Notebook 4: Final Submission")